# Explainable Android Malware Detection - Random Forest Baseline

This notebook covers the training and evaluation of the baseline Random Forest model on the `cleaned_dataset.csv`.
This establishes our predictive baseline before introducing obfuscated data.

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import time
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Create directories for saving models and data mappings
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Optional: SHAP for explainability (ensure shap is installed: pip install shap)
try:
    import shap
except ImportError:
    print("SHAP is not installed. You can install it using: pip install shap")

### Load the Cleaned Dataset

In [ ]:
# Step 2: Load the dataset
data_path = 'cleaned_dataset.csv'
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['Label'].value_counts(normalize=True) * 100}")
df.head()

### Data Preprocessing & Splitting
We separate the features from the target variable (`Label`) and preserve the `apk_id` column to map test samples back later.

In [ ]:
# Step 3: Separate features (X), target (y), and IDs
id_col = 'apk_id'
ids = df[id_col]
X = df.drop(['Label', id_col], axis=1)
y = df['Label']

# Step 4: Train-Test Split (80% training, 20% testing)
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

print(f"\nClass distribution in training set (Imbalance Check):\n{y_train.value_counts()}")

# Save the test set ID mapping for later obfuscation matching
test_mapping_path = 'data/rf_test_set_ids.csv'
pd.DataFrame({'apk_id': id_test, 'label': y_test}).to_csv(test_mapping_path, index=False)
print(f"\nTest set IDs saved to {test_mapping_path}")

### Model Initialization, Training and Persistence
We initialize the Random Forest model, adding `class_weight='balanced'` to counter the malware vs benign class imbalance.

In [ ]:
# Step 5: Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,       # Number of trees in the forest
    max_depth=None,         # Let trees expand fully to capture complex patterns
    random_state=42,        # Lock randomness for fair comparison later
    n_jobs=-1,              # Use all available CPU cores for faster training
    class_weight='balanced' # Automatically adjust weights inversely proportional to class frequencies
)

print("Training Random Forest...")
start_time = time.time()
rf_model.fit(X_train, y_train)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds.")

# Step 6: Save the trained model to disk
model_path = 'models/rf_baseline.pkl'
joblib.dump(rf_model, model_path)
print(f"Model saved to {model_path}")

# Evaluate model size
size_kb = os.path.getsize(model_path) / 1024
print(f"Model size: {size_kb:.2f} KB")

### Performance Evaluation
Now we evaluate the model on the unseen test set.

In [ ]:
# Step 7: Make predictions and evaluate
print("Evaluating model on test set...")
start_time = time.time()
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
inference_time = time.time() - start_time

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- Model Performance ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Inference time for {len(y_test)} samples: {inference_time:.4f} seconds")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malware (1)']))

### Confusion Matrix Visualization

In [ ]:
# Step 8: Plot the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malware'], yticklabels=['Benign', 'Malware'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Random Forest Confusion Matrix')
plt.show()

### Feature Importance (Basic)
Before diving into SHAP, we can look at the built-in feature importance provided by Random Forest (Gini importance).

In [ ]:
# Step 9: Plot Top 10 Feature Importances (Built-in)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

top_k = 10
top_features = X.columns[indices][:top_k]
top_importances = importances[indices][:top_k]

plt.figure(figsize=(10,6))
plt.barh(range(top_k), top_importances[::-1], align='center')
plt.yticks(range(top_k), top_features[::-1])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features (Random Forest Built-in)')
plt.show()